In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# Perren+2023 catalog

In [ ]:
import pandas as pd

catalog_name = 'Perren2023'
df = pd.read_csv('../data/Perren2023/UCC_cat.csv')
df.tail()

In [ ]:
df = df.rename({"name": "Name",
                #"Source": "GaiaDR3",
                "probs": "Prob",
                "RA_ICRS": "ra",
                "DE_ICRS": "dec",
                "pmRA": "pmra",
                "pmDE": "pmdec",
                "Plx": "parallax",
                "RV": "radial_velocity"
                }, axis=1)
df

In [ ]:
df_mem = pd.read_parquet('../data/Perren2023/UCC_members.parquet')
df_mem.tail()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({#"name": "Name",
                        "GaiaDR3": "source_id",
                        #"probs": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem.shape

In [ ]:
from wakai.catalog import get_nexsci_data

table='pscomppars'
nexsci = get_nexsci_data(table, clobber=True)
nexsci.tail()

In [ ]:
nexsci.shape

In [ ]:
if table=='ps':
    nexsci_hosts = nexsci[nexsci['default_flag']==1]
else:
    nexsci_hosts = nexsci.copy()
nexsci_hosts.shape

In [ ]:
nexsci.hostname.unique().shape

# cross-match nexsci with catalog

In [ ]:
import pandas as pd

if False:
    #from astroquery.xmatch
    matches = pd.read_csv('../data/nexsci_GaiaDR3_xmatch.csv')
if True:
    #from vizier query/exofop; not yet available
    matches = pd.read_csv('../data/nexsci_with_Gaiaid.csv')
    matches['GaiaDR3'] = matches['GAIA']
matches.head()

In [ ]:
final = pd.merge(df_mem, matches, left_on='source_id', right_on='GaiaDR3')
final.head()

In [ ]:
nexsci.query("hostname=='EPIC 211822797'")

In [ ]:
final[['pl_name','Name']]

In [ ]:
final.hostname.unique().shape

In [ ]:
if table=='ps':
    final_default = final[final['default_flag']==1]
else:
    final_default = final.copy()
final_default.shape

In [ ]:
final_default.sort_values(by='pl_name')\
    .to_csv(f'../data/{catalog_name}/nexsci_in_{catalog_name}.csv', 
            index=False)

# nearby

In [ ]:
from astropy.coordinates import SkyCoord

# epoch already in J2016.0
nexsci_coords = SkyCoord(nexsci['ra'].values, nexsci['dec'].values, 
                      distance=nexsci['sy_dist'].values, unit=('deg','deg','pc')) 
nexsci_coords

In [ ]:
from astropy.coordinates import SkyCoord, Distance
import astropy.units as u

coords1 = SkyCoord(ra=nexsci_coords.ra, dec=nexsci_coords.dec, distance=nexsci_coords.distance, frame='icrs')
coords2 = SkyCoord(ra=df_mem['ra'].values*u.degree, 
                   dec=df_mem['dec'].values*u.degree, 
                   distance=Distance(parallax=df_mem['parallax'].values* u.mas, allow_negative=True),
                   frame='icrs') 

# `match_to_catalog_sky` finds the nearest match in coords2 for each coordinate in coords1
# It returns the indices into the second catalog, the angular separation, and 3D separation (if applicable)
idx, d2d, d3d = coords1.match_to_catalog_sky(coords2)

# Add separation distance to df1
nexsci['sep2d_arcsec'] = d2d.to(u.arcsec).value
nexsci['sep3d_pc'] = d3d.to(u.pc).value
nexsci.sort_values('sep2d_arcsec').head()

In [ ]:
sep_2d = 3.0 * u.arcsec
sep_3d = 10 * u.pc #

# Add matching information to the original dataframes
match_mask_2d = d2d < sep_2d 
match_mask_3d = d3d < sep_3d

nexsci[match_mask_2d].shape, nexsci[match_mask_3d].shape

In [ ]:
from wakai.catalog import get_cluster_3d_size

# test on praesepe
d = df_mem.groupby('Name').get_group('ngc2632')
get_cluster_3d_size(d)

In [ ]:
# apply func to all clusters
df_size = df_mem.groupby('Name').apply(get_cluster_3d_size)

In [ ]:
df = df.merge(df_size.reset_index(name='extent_pc'), on='Name')
df.head()

# match by 3d separation with cluster core

In [ ]:
from astropy.coordinates import SkyCoord, Distance

coords1 = SkyCoord(ra=nexsci_coords.ra, dec=nexsci_coords.dec, distance=nexsci_coords.distance, frame='icrs')
coords2 = SkyCoord(ra=df['ra'].values*u.degree, 
                   dec=df['dec'].values*u.degree, 
                   distance=Distance(parallax=df['parallax'].values* u.mas, allow_negative=True),
                   frame='icrs') 

idx, d2d, d3d = coords1.match_to_catalog_sky(coords2)

nexsci['sep2d_arcsec'] = d2d.to(u.arcsec).value 
nexsci['sep3d_pc'] = d3d.to(u.pc).value
nexsci.sort_values('sep2d_arcsec').head()

In [ ]:
# Add a prefix to matched columns to avoid confusion (optional but recommended)
col_prefix = 'matched_'
df_matched_nexsci = df.iloc[idx].copy().add_prefix(col_prefix)

# Concatenate horizontally
df_match = pd.concat([nexsci.reset_index(drop=True), 
                      df_matched_nexsci.reset_index(drop=True)
                      ], axis=1)

In [ ]:
# make sure the 3d sep from cluster core is smaller the spatial extent of that cluster
idx2 = df_match[f'{col_prefix}extent_pc'] > df_match['sep3d_pc']
df_match[idx2]

In [ ]:
import numpy as np

df_match['pm_diff'] =  np.sqrt((df_match[f'{col_prefix}pmra'] - df_match[f'{col_prefix}pmdec'])**2 + 
                               (df_match['sy_pmra'] - df_match['sy_pmdec'])**2)

In [ ]:
idx = df_match.columns.str.contains('gaia')
df_match.columns[idx]

In [ ]:
idx3 = df_match[idx2]['pm_diff'] < 10
cols = ['hostname','matched_Name','matched_Age_[Myr]',
        'matched_extent_pc','sep3d_pc','pm_diff','tic_id','gaia_dr3_id']
df_final = df_match[idx2].loc[idx3,cols].reset_index(drop=True)
df_final

In [ ]:
df_match['source_id'] = df_match.gaia_dr3_id.apply(lambda x: str(x).split()[-1] if len(str(x).split())>1 else x)

In [ ]:
df_match['parallax'] = df_match.matched_parallax.copy()
df_match['pmra'] = df_match.matched_pmra.copy()
df_match['pmdec'] = df_match.matched_pmdec.copy()
df_match['radial_velocity'] = df_match.matched_Rv.copy()

In [ ]:
from wakai.catalog import plot_rdp_pmrv

hostname = 'TOI-837'
df_target = df_match.query("hostname==@hostname").squeeze()
target_gaiaid = int(df_target.squeeze().source_id)
age = df_target["matched_Age_[Myr]"]
cname = df_target["matched_Name"]
d = df_mem.groupby("Name").get_group(cname)

fig = plot_rdp_pmrv(d,
                   color='Prob', 
                    target_gaiaid=target_gaiaid,
                    match_id=False,
                   df_target=df_target,
                   target_color="C3",
                   marker='.',
                   cmap="Blues");
fig.suptitle(f"{hostname} in {cname}")

In [ ]:
from wakai.catalog import plot_rdp_pmrv
from wakai.catalog import plot_xyz_uvw

hostname = 'TOI-837'
df_target = df_match.query("hostname==@hostname").squeeze()
target_gaiaid = int(df_target.squeeze().source_id)
age = df_target["matched_Age_[Myr]"]
cname = df_target["matched_Name"]
d = df_mem.groupby("Name").get_group(cname)

fig = plot_rdp_pmrv(d,
                   color='Prob', 
                    target_gaiaid=target_gaiaid,
                    match_id=False,
                   df_target=df_target,
                   target_color="C3",
                   marker='.',
                   cmap="Blues");
fig.suptitle(f"{hostname} in {cname}")

In [ ]:
import matplotlib.pyplot as pl
from wakai.catalog import plot_rdp_pmrv
from wakai.catalog import plot_xyz_uvw

outdir = '../output/nexsci_near_clusters'
groups = df_mem.groupby("Name")
for i,row in df_final.sort_values('hostname', ascending=False).iterrows():
    try:
        hostname = row.hostname
        df_target = df_match.drop_duplicates(subset='hostname', keep='first').query("hostname==@hostname").squeeze()
        target_gaiaid = int(df_target.squeeze().source_id)
        age = df_target["matched_Age_[Myr]"]
        cname = df_target["matched_Name"]
        d = df_mem.groupby("Name").get_group(cname)
        if len(d.radial_velocity.dropna())<10:
            continue

        fp = f'{outdir}/{hostname}_in_{cname}_{age:.0f}Myr'
        if True:
            fig = plot_rdp_pmrv(d,
                               color='Prob', 
                               target_gaiaid=target_gaiaid,
                               match_id=False,
                               df_target=df_target,
                               target_color="C3",
                               marker='.',
                               cmap="Blues");
            fp=fp+'.png'
        else:
            fig, ax = plot_xyz_uvw(d, 
                               color='Prob', 
                               df_target=df_target,
                               target_color="C3",
                               target_ms=20,
                               marker='.',
                               cmap="Blues",
                               verbose=False
                              );
            fp=fp+'_xyz.png'
        fig.suptitle(f"{hostname} in {cname} ({age:.0f} Myr)")
        print(f"https://ucc.ar/_clusters/{cname}")
        fig.savefig(fp, bbox_inches='tight', dpi=200)
        # pl.show()
    except Exception as e:
        print(e)